# 02 — Text Analytics Pipeline: F0 → F5

**DS 5001 — Exploratory Text Analytics Final Project**

This notebook runs the full text analytics pipeline on the papal encyclicals corpus:

| Stage | Description |
|-------|-------------|
| F0 → F1 | Raw text → paragraphs indexed by document hierarchy |
| F1 → F2 | Tokenization → LIBRARY, TOKEN, VOCAB tables (STADM) |
| F2 → F3 | NLP annotations: POS, lemma, stopwords, sentiment |
| F3 → F4 | TFIDF vectorization |
| F4 → F5 | PCA, LDA topic models, word2vec embeddings |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.pipeline import (
    build_f1_corpus, build_f2_tables, build_f3_annotations,
    build_f4_tfidf, build_f5_models, save_tables,
    PROCESSED_DIR
)

## F0 → F1: Machine Learning Corpus Format

Split raw documents into paragraphs (minimum discursive units),
indexed by document content hierarchy.

In [2]:
corpus = build_f1_corpus(english_only=True)
print(f"Corpus shape: {corpus.shape}")
print(f"Documents: {corpus['doc_id'].nunique()}")
print(f"Paragraphs: {len(corpus)}")
corpus.head()

2026-03-30 11:58:59,610 [INFO] Building F1 corpus from raw text files...
2026-03-30 11:59:05,167 [INFO] F1 corpus: 42266 paragraphs from 551 documents


Corpus shape: (42266, 6)
Documents: 551
Paragraphs: 42266


,doc_id,pope,title,year,para_num,para_text
0,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,0,INTRODUCTION
1,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,1,This council was summoned by pope Julius II by...
2,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,2,There were twelve sessions. The first five of ...
3,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,3,"All the decrees of this council, at which the ..."
4,church_councils__the_fifth_general_council_of_...,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,4,The decisions on the reform of the curia produ...


## F1 → F2: STADM Tables

Tokenize into LIBRARY (doc metadata), TOKEN (every word), VOCAB (unique terms).

In [3]:
LIBRARY, TOKEN, VOCAB = build_f2_tables(corpus)
print("LIBRARY:"); display(LIBRARY.head())
print("\nTOKEN:"); display(TOKEN.head(10))
print("\nVOCAB (top 20):"); display(VOCAB.head(20))

2026-03-30 11:59:05,206 [INFO] Building F2 STADM tables (LIBRARY, TOKEN, VOCAB)...
Tokenizing: 100%|██████████| 42266/42266 [00:40<00:00, 1046.87it/s]
2026-03-30 12:00:26,884 [INFO] LIBRARY: 551 documents
2026-03-30 12:00:26,885 [INFO] TOKEN:   4212475 tokens
2026-03-30 12:00:26,886 [INFO] VOCAB:   84950 unique terms


LIBRARY:


,pope,title,year,n_paragraphs,n_chars,n_tokens
doc_id,,,,,,
church_councils__the_fifth_general_council_of_the_lateran__1512_17,Church Councils,"The Fifth General Council of the Lateran, 1512-17",1512,196,211930,40440
church_councils__the_first_general_council_of_constantinople__381,Church Councils,"The First General Council of Constantinople, 381",,51,22891,4427
church_councils__the_first_general_council_of_lyons__1245,Church Councils,"The First General Council of Lyons, 1245",1245,94,77512,15108
church_councils__the_first_general_council_of_nicaea__325,Church Councils,"The First General Council of Nicaea, 325",,20,6097,1169
church_councils__the_first_general_council_of_the_lateran__1123,Church Councils,"The First General Council of the Lateran, 1123",,37,16643,3527



TOKEN:


,doc_id,para_num,sent_num,token_num,token_str,term_str
token_id,,,,,,
0,church_councils__the_fifth_general_council_of_...,0,0,0,INTRODUCTION,introduction
1,church_councils__the_fifth_general_council_of_...,1,0,0,This,this
2,church_councils__the_fifth_general_council_of_...,1,0,1,council,council
3,church_councils__the_fifth_general_council_of_...,1,0,2,was,was
4,church_councils__the_fifth_general_council_of_...,1,0,3,summoned,summoned
5,church_councils__the_fifth_general_council_of_...,1,0,4,by,by
6,church_councils__the_fifth_general_council_of_...,1,0,5,pope,pope
7,church_councils__the_fifth_general_council_of_...,1,0,6,Julius,julius
8,church_councils__the_fifth_general_council_of_...,1,0,7,II,ii



VOCAB (top 20):


,n,df,idf
term_str,,,
the,283582,480,0.137949
",",246021,549,0.003636
of,184101,481,0.135868
and,130699,479,0.140034
.,130501,545,0.010949
to,111606,479,0.140034
in,85845,544,0.012786
is,47132,476,0.146317
a,46484,545,0.010949


## F2 → F3: NLP Annotations

Add POS tags, lemmas, stopword flags, and VADER sentiment scores.

In [4]:
LIBRARY, TOKEN, VOCAB = build_f3_annotations(TOKEN, VOCAB, LIBRARY)
print("TOKEN with annotations:"); display(TOKEN.head(10))
print("\nVOCAB with annotations:"); display(VOCAB.head(10))
print("\nLIBRARY sentiment:"); display(LIBRARY[['pope','title','sentiment_compound']].head())

2026-03-30 12:00:26,921 [INFO] Building F3 NLP annotations...
2026-03-30 12:00:33,910 [INFO]   POS tagging...
POS tagging: 100%|██████████| 146074/146074 [02:41<00:00, 905.23it/s] 
2026-03-30 12:03:19,195 [INFO]   Lemmatizing...
2026-03-30 12:07:12,194 [INFO]   Updating VOCAB...
2026-03-30 12:07:43,815 [INFO]   Computing VADER sentiment for vocab...
2026-03-30 12:07:46,396 [INFO]   Computing document-level sentiment...
2026-03-30 12:12:55,936 [INFO]   F3 annotations complete


TOKEN with annotations:


,doc_id,para_num,sent_num,token_num,token_str,term_str,pos,lemma,is_stop,is_alpha
token_id,,,,,,,,,,
0,church_councils__the_fifth_general_council_of_...,0,0,0,INTRODUCTION,introduction,NN,introduction,False,True
1,church_councils__the_fifth_general_council_of_...,1,0,0,This,this,DT,this,True,True
2,church_councils__the_fifth_general_council_of_...,1,0,1,council,council,NN,council,False,True
3,church_councils__the_fifth_general_council_of_...,1,0,2,was,was,VBD,be,True,True
4,church_councils__the_fifth_general_council_of_...,1,0,3,summoned,summoned,VBN,summon,False,True
5,church_councils__the_fifth_general_council_of_...,1,0,4,by,by,IN,by,True,True
6,church_councils__the_fifth_general_council_of_...,1,0,5,pope,pope,NN,pope,False,True
7,church_councils__the_fifth_general_council_of_...,1,0,6,Julius,julius,NNP,julius,False,True
8,church_councils__the_fifth_general_council_of_...,1,0,7,II,ii,NNP,ii,False,True



VOCAB with annotations:


,n,df,idf,pos,lemma,is_stop,vader_neg,vader_neu,vader_pos,vader_compound
term_str,,,,,,,,,,
the,283582,480,0.137949,DT,the,True,0.0,1.0,0.0,0.0
",",246021,549,0.003636,",",",",False,0.0,0.0,0.0,0.0
of,184101,481,0.135868,IN,of,True,0.0,1.0,0.0,0.0
and,130699,479,0.140034,CC,and,True,0.0,1.0,0.0,0.0
.,130501,545,0.010949,.,.,False,0.0,0.0,0.0,0.0
to,111606,479,0.140034,TO,to,True,0.0,1.0,0.0,0.0
in,85845,544,0.012786,IN,in,True,0.0,1.0,0.0,0.0
is,47132,476,0.146317,VBZ,be,True,0.0,1.0,0.0,0.0
a,46484,545,0.010949,DT,a,True,0.0,0.0,0.0,0.0



LIBRARY sentiment:


,pope,title,sentiment_compound
doc_id,,,
church_councils__the_fifth_general_council_of_the_lateran__1512_17,Church Councils,"The Fifth General Council of the Lateran, 1512-17",0.9996
church_councils__the_first_general_council_of_constantinople__381,Church Councils,"The First General Council of Constantinople, 381",0.9997
church_councils__the_first_general_council_of_lyons__1245,Church Councils,"The First General Council of Lyons, 1245",0.9998
church_councils__the_first_general_council_of_nicaea__325,Church Councils,"The First General Council of Nicaea, 325",0.9995
church_councils__the_first_general_council_of_the_lateran__1123,Church Councils,"The First General Council of the Lateran, 1123",-0.6946


## F3 → F4: TFIDF Vectorization

Compute TF-IDF scores and build the document-term matrix.

In [5]:
LIBRARY, TOKEN, VOCAB, TFIDF_DTM = build_f4_tfidf(TOKEN, VOCAB, LIBRARY)
print(f"Document-term matrix: {TFIDF_DTM.shape}")
print(f"\nTop TFIDF terms per document (first 3 docs):")
for doc_id in TFIDF_DTM.index[:3]:
    top = TFIDF_DTM.loc[doc_id].nlargest(5)
    print(f"  {doc_id[:40]}: {', '.join(top.index)}")

2026-03-30 12:12:56,215 [INFO] Building F4 TFIDF features...
2026-03-30 12:13:00,348 [INFO]   Creating document-term matrix...
2026-03-30 12:18:17,225 [INFO]   TFIDF DTM shape: (551, 29674)


Document-term matrix: (551, 29674)

Top TFIDF terms per document (first 3 docs):
  church_councils__the_fifth_general_counc: benefices, council, cardinals, prelates, approval
  church_councils__the_first_general_counc: constantinople, contents, arians, nicene, table
  church_councils__the_first_general_counc: excommunication, prelates, coll, plaintiff, empire


## F4 → F5: Unsupervised Models

Fit PCA, LDA, and word2vec models.

In [7]:
f5_results = build_f5_models(
    LIBRARY, TOKEN, VOCAB, TFIDF_DTM,
    n_components=10, n_topics=10, w2v_dim=100
)

print("PCA - documents x components:")
display(f5_results['DOC_PCA'].head())
print(f"\nExplained variance: {f5_results['explained_variance'].sum():.1%}")

print("\nLDA - documents x topics:")
display(f5_results['DOC_TOPICS'].head())

print(f"\nword2vec embeddings: {f5_results['EMBEDDINGS'].shape}")

2026-03-30 13:12:52,171 [INFO] Building F5 unsupervised models...
2026-03-30 13:12:53,482 [INFO]   PCA with 10 components...
2026-03-30 13:12:54,486 [INFO]     Explained variance: 7.98%
2026-03-30 13:12:54,487 [INFO]   LDA with 10 topics...
2026-03-30 13:16:48,819 [INFO]     Topic 0: church, bishop, apostolic, rite, priest, sacred, see, holy, congregation, cardinal
2026-03-30 13:16:48,822 [INFO]     Topic 1: human, social, life, people, work, world, good, society, need, one
2026-03-30 13:16:48,826 [INFO]     Topic 2: truth, god, one, must, man, faith, moral, human, life, reason
2026-03-30 13:16:48,830 [INFO]     Topic 3: god, love, christ, life, man, church, spirit, one, jesus, word
2026-03-30 13:16:48,832 [INFO]     Topic 4: family, law, marriage, god, church, state, right, men, child, authority
2026-03-30 13:16:48,835 [INFO]     Topic 5: di, che, la, il, non, per, si, le, del, della
2026-03-30 13:16:48,838 [INFO]     Topic 6: church, god, catholic, great, men, brother, may, people, v

PCA - documents x components:


,PC0,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9
doc_id,,,,,,,,,,
church_councils__the_fifth_general_council_of_the_lateran__1512_17,-9.267815,-1.091718,-6.828320,1.930256,3.926128,-0.307146,-0.719624,0.854905,3.054655,-1.065780
church_councils__the_first_general_council_of_constantinople__381,-8.084910,-1.388035,-5.573933,0.391458,2.567515,0.083561,-0.617016,0.600846,2.667449,-0.769960
church_councils__the_first_general_council_of_lyons__1245,-9.173552,-1.280868,-9.610771,1.339964,5.554314,-0.430323,-1.092931,1.345112,3.637338,-1.121728
church_councils__the_first_general_council_of_nicaea__325,-5.317552,-0.037002,-7.663905,0.339926,0.800500,0.223540,-0.334544,0.149125,1.421995,0.126602
church_councils__the_first_general_council_of_the_lateran__1123,-4.896322,2.141564,-9.862244,1.796193,2.288487,0.591818,1.243403,-0.670487,4.546367,-1.284852



Explained variance: 8.0%

LDA - documents x topics:


,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9
doc_id,,,,,,,,,,
church_councils__the_fifth_general_council_of_the_lateran__1512_17,0.000509,0.008889,0.026242,0.000006,0.000006,0.000317,0.081144,0.001319,0.000006,0.881560
church_councils__the_first_general_council_of_constantinople__381,0.000063,0.000063,0.000063,0.092033,0.000063,0.000063,0.053490,0.000063,0.034592,0.819508
church_councils__the_first_general_council_of_lyons__1245,0.000018,0.004968,0.000018,0.000018,0.000018,0.000018,0.098907,0.000018,0.000018,0.895998
church_councils__the_first_general_council_of_nicaea__325,0.068543,0.000248,0.000248,0.104622,0.000248,0.000248,0.070014,0.000248,0.000248,0.755332
church_councils__the_first_general_council_of_the_lateran__1123,0.233820,0.000095,0.000095,0.006044,0.008414,0.004371,0.000095,0.018615,0.000095,0.728357



word2vec embeddings: (15794, 100)


## Save All Tables

In [8]:
save_tables(LIBRARY, TOKEN, VOCAB, TFIDF_DTM, f5_results)
print("All tables saved!")

# List output files
for f in sorted(PROCESSED_DIR.glob('*.csv')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}: {size_kb:.0f} KB")

2026-03-30 13:18:40,285 [INFO] Saving tables to CSV...
2026-03-30 13:19:44,846 [INFO] All tables saved to C:\Users\harrisrc\OneDrive - Chesterfield County VA\Documents\MSDS\encyclicals\data\processed/


All tables saved!
  dead_link_replacements.csv: 10 KB
  DOC_PCA.csv: 126 KB
  DOC_TOPICS.csv: 135 KB
  EMBEDDINGS.csv: 18359 KB
  explained_variance.csv: 0 KB
  LIBRARY.csv: 64 KB
  LOADINGS.csv: 6799 KB
  TFIDF_DTM.csv: 75217 KB
  TOKEN.csv: 447232 KB
  TOPIC_TERMS.csv: 977 KB
  VOCAB.csv: 7710 KB
